Import Libraries

In [22]:
from langgraph.graph import START, END, StateGraph
from langchain_groq import ChatGroq
from typing import TypedDict,Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [23]:
load_dotenv()
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

create schema of sentiment

In [24]:
class SentimentSchema(BaseModel):
    sentiment:Literal["Positive","Negative"]=Field(description="Sentiment oof the review")

create schema of Diagnosis

In [25]:
class DiagnosisSchema(BaseModel):
    issue_type:Literal["UX","Performance","Bug","Support","Other"]=Field(description="The category of issue mentioned in review")
    tone:Literal["Frustrated","Angry","Disappointed","Calm"]=Field(description="Emotional tone expressed by user in review")
    urgency:Literal["Low","High","Medium"]=Field(description="How urgent or critical issue appears to be")

create structured models

In [26]:
structure_Sentiment_Model = llm.with_structured_output(SentimentSchema)
structure_Diagnosis_Model = llm.with_structured_output(DiagnosisSchema)

Create State

In [27]:
class ReviewState(TypedDict):
    review:str
    sentiment:Literal["Positive","Negative"]
    diagnosis:dict
    response:str

create graph

In [28]:
def find_sentiment(state:ReviewState):
    prompt=f"For the given revview find out the sentiment {state['review']}"
    sentiment = structure_Sentiment_Model.invoke(prompt)
    return {'sentiment':sentiment}

In [29]:
def check_sentiment(state:ReviewState)->Literal["Positive_Response","Run_Diagnosis"]:
    if state["sentiment"]=="Positive":
        return "Positive_Response"
    else:
        return "Run_Diagnosis"

In [30]:
def positive_response(state:ReviewState):
    prompt=f"Write a warm than you message on the given review \n \n {state['review']} also ask customer to provide feedback on website"
    response = llm.invoke(prompt)
    return {'response':response}

In [31]:
def run_diagnosis(state:ReviewState):
    prompt=f"Diagnose this negative review: \n\n {state['review']} and return the issue type, tone and urgency"
    response = structure_Diagnosis_Model.invoke(prompt)
    return {'diagnosis':response.model_dump()}

In [37]:
def negative_response(state:ReviewState):
    diagnosis = state['diagnosis']
    prompt =f"You are a support assistant. The user had a {diagnosis['issue_type']} issue sounded {diagnosis['tone']} and marked urgency as {diagnosis['urgency']} \n write an empathetic, helpful resolution message"
    response = llm.invoke(prompt).content
    return {'response':response}

In [38]:
graph=StateGraph(ReviewState)
graph.add_node("Find_Sentiment",find_sentiment)
graph.add_node("Run_Diagnosis",run_diagnosis)
graph.add_node("Positive_Response",positive_response)
graph.add_node("Negative_Response",negative_response)

graph.add_edge(START,"Find_Sentiment")
graph.add_conditional_edges("Find_Sentiment",check_sentiment)
graph.add_edge("Positive_Response",END)
graph.add_edge("Run_Diagnosis","Negative_Response")
graph.add_edge("Negative_Response",END)

In [39]:
workflow = graph.compile()

In [40]:
initial_state={'review':"I've been trying to logged in for over an hour now and the app keeps freezing on authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptible, wspecially when it affects basic functionality"}

In [41]:
final_state=workflow.invoke(initial_state)

In [42]:
print(final_state)

{'review': "I've been trying to logged in for over an hour now and the app keeps freezing on authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptible, wspecially when it affects basic functionality", 'sentiment': SentimentSchema(sentiment='Negative'), 'diagnosis': {'issue_type': 'Bug', 'tone': 'Frustrated', 'urgency': 'High'}, 'response': "Here's a potential response:\n\n**Subject: Urgent: Help with Bug Issue**\n\nDear [User's Name],\n\nI apologize for the frustrating experience you're having with the bug issue, and I appreciate you bringing it to my attention. I understand that you've marked the urgency as High, and I'm here to help resolve it as quickly as possible.\n\nI'm sorry to hear that you're experiencing [briefly mention the bug issue]. I can imagine how frustrating it must be to encounter this problem, and I'm committed to helping you get back on track.\n\nTo better assist you, could you please provide more details about the issue you